# PoliMillionaire Game Tests

Run live games with selected local models, one after another.

No generated-answer API is used.


## Setup

Find the repo and local model cache.


In [1]:
import gc
import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "polimillionaire" / "__init__.py").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the project folder.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
for path in [REPO_ROOT / "src", REPO_ROOT]:
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

os.environ.setdefault("HF_HOME", str(REPO_ROOT / "data" / "hf_home"))
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", str(REPO_ROOT / "data" / "hf_home" / "hub"))

print("Repo:", REPO_ROOT)
print("HF cache:", os.environ["HF_HOME"])


Repo: C:\Users\sjuan\Desktop\NLP\project\nlp-polimillionaire
HF cache: C:\Users\sjuan\Desktop\NLP\project\nlp-polimillionaire\data\hf_home


## Settings

Pick models here. Each selected model gets its own game run.


In [2]:
RUN_API_CHECK = True
RUN_LIVE_GAME = True
PROMPT_FOR_CREDENTIALS = False

API_URL = "http://131.175.15.22:51111/"
COMPETITION_ID = 0

MODELS_TO_RUN = [
    {"label": "Gemma 4 E2B", "model_id": "google/gemma-4-E2B-it", "run": True},
    {"label": "Gemma 4 E4B", "model_id": "google/gemma-4-E4B-it", "run": True},
]

SAFE_DELAY_SECONDS = 2.0
ANSWER_TIMEOUT_SECONDS = 25.0

print("API check:", RUN_API_CHECK)
print("Live game:", RUN_LIVE_GAME)
print("Competition:", COMPETITION_ID)
print("Models:")
for item in MODELS_TO_RUN:
    print("-", item["label"], item["model_id"], "run=", item["run"])


API check: True
Live game: True
Competition: 0
Models:
- Gemma 4 E2B google/gemma-4-E2B-it run= True
- Gemma 4 E4B google/gemma-4-E4B-it run= True


## Login

Use environment variables, Colab secrets, or prompt mode.


In [3]:
import getpass
from millionaire_client import AuthenticationError, MillionaireClient

USERNAME = os.environ.get("POLIMILLIONAIRE_USERNAME")
PASSWORD = os.environ.get("POLIMILLIONAIRE_PASSWORD")

try:
    from google.colab import userdata

    USERNAME = USERNAME or userdata.get("POLIMILLIONAIRE_USERNAME")
    PASSWORD = PASSWORD or userdata.get("POLIMILLIONAIRE_PASSWORD")
except Exception:
    pass

if PROMPT_FOR_CREDENTIALS and not USERNAME:
    USERNAME = input("PoliMillionaire username: ").strip()
if PROMPT_FOR_CREDENTIALS and not PASSWORD:
    PASSWORD = getpass.getpass("PoliMillionaire password: ")

print("Username configured:", bool(USERNAME))
print("Password configured:", bool(PASSWORD))


Username configured: True
Password configured: True


## Competitions

Turn on `RUN_API_CHECK` to list games.


In [4]:
client = None

if RUN_API_CHECK:
    if not USERNAME or not PASSWORD:
        raise RuntimeError("Set credentials first.")
    try:
        client = MillionaireClient(API_URL)
        user = client.login(USERNAME, PASSWORD)
        print("Logged in as", user.username)
        for competition in client.competitions.list_all():
            print(competition.id, competition.name, competition.max_levels)
    except AuthenticationError as exc:
        client = None
        print("Login failed:", exc)
else:
    print("API check skipped.")


Logged in as sanchez
0 Entertainment 15
1 Ancient History and Politics 15
2 Science and Nature 15
3 Maths 15


## Run Selected Models

Each model warms up, plays if enabled, then unloads.


In [5]:
from datetime import datetime, timezone

from polimillionaire.runner import GameRunner, RunLogger
from polimillionaire.strategies import GemmaLLMConfig, GemmaStrategy
from polimillionaire.types import AnswerOption, Question

warmup_question = Question(
    id=1,
    text="What is 2 + 2?",
    options=[
        AnswerOption(0, "3"),
        AnswerOption(1, "4"),
        AnswerOption(2, "5"),
        AnswerOption(3, "22"),
    ],
)


def clear_model_memory():
    gc.collect()
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        if hasattr(torch, "mps") and hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            torch.mps.empty_cache()
    except Exception as exc:
        print("Cleanup warning:", exc)


def make_strategy(model_id: str) -> GemmaStrategy:
    config = GemmaLLMConfig(
        model_id=model_id,
        inference_backend="auto_model",
        device_map="auto",
        dtype="auto",
        max_new_tokens=8,
        temperature=0.0,
        do_sample=False,
        num_beams=1,
        seed=42,
        generation_max_time_seconds=18.0,
        timeout_seconds=120.0,
    )
    return GemmaStrategy(model_config=config)


def clean_name(label: str) -> str:
    return "_".join(label.lower().split())


if RUN_LIVE_GAME and (not USERNAME or not PASSWORD):
    raise RuntimeError("Set credentials first.")
if RUN_LIVE_GAME and client is None:
    client = MillionaireClient(API_URL)
    client.login(USERNAME, PASSWORD)

run_results = []
for item in MODELS_TO_RUN:
    if not item.get("run", False):
        print("Skipped", item["label"])
        continue

    strategy = None
    try:
        print()
        print("Model:", item["label"])
        strategy = make_strategy(item["model_id"])

        warmup = strategy.answer(warmup_question)
        print("warmup option:", warmup.option_id, warmup.answer_text)
        print("fallback:", warmup.metadata.get("fallback"))
        print("device:", warmup.metadata.get("device"))
        if warmup.metadata.get("fallback"):
            raise RuntimeError(f"Warm-up failed for {item['label']}.")

        result = {"label": item["label"], "model_id": item["model_id"], "warmup_option": warmup.option_id}

        if RUN_LIVE_GAME:
            run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
            log_path = REPO_ROOT / "results" / "runs" / f"{clean_name(item['label'])}_{run_id}.jsonl"
            runner = GameRunner(
                client=client,
                safe_delay_seconds=SAFE_DELAY_SECONDS,
                answer_timeout_seconds=ANSWER_TIMEOUT_SECONDS,
                logger=RunLogger(log_path),
            )
            game = runner.play(COMPETITION_ID, strategy)
            result.update(
                {
                    "current_level": game.current_level,
                    "earned_amount": game.earned_amount,
                    "log_path": str(log_path),
                }
            )
            print("Reached level:", game.current_level)
            print("Earned amount:", game.earned_amount)
            print("Log path:", log_path)
        else:
            print("Live game skipped for", item["label"])

        run_results.append(result)
    finally:
        del strategy
        clear_model_memory()
        print("Cleared model memory after", item["label"])

run_results



Model: Gemma 4 E2B


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

warmup option: 1 4
fallback: False
device: cuda:0
Reached level: 1
Earned amount: 0
Log path: C:\Users\sjuan\Desktop\NLP\project\nlp-polimillionaire\results\runs\gemma_4_e2b_20260504_115907.jsonl
Cleared model memory after Gemma 4 E2B

Model: Gemma 4 E4B


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


warmup option: 1 4
fallback: False
device: {'model.vision_tower': 0, 'model.language_model.embed_tokens': 0, 'lm_head': 0, 'model.language_model.layers.0': 0, 'model.language_model.layers.1': 0, 'model.language_model.layers.2': 0, 'model.language_model.layers.3': 0, 'model.language_model.layers.4': 0, 'model.language_model.layers.5': 0, 'model.language_model.layers.6': 0, 'model.language_model.layers.7': 0, 'model.language_model.layers.8': 0, 'model.language_model.layers.9': 0, 'model.language_model.layers.10': 0, 'model.language_model.layers.11': 0, 'model.language_model.layers.12': 0, 'model.language_model.layers.13': 0, 'model.language_model.layers.14': 'cpu', 'model.language_model.layers.15': 'cpu', 'model.language_model.layers.16': 'cpu', 'model.language_model.layers.17': 'cpu', 'model.language_model.layers.18': 'cpu', 'model.language_model.layers.19': 'cpu', 'model.language_model.layers.20': 'cpu', 'model.language_model.layers.21': 'cpu', 'model.language_model.layers.22': 'cpu', 

[{'label': 'Gemma 4 E2B',
  'model_id': 'google/gemma-4-E2B-it',
  'warmup_option': 1,
  'current_level': 1,
  'earned_amount': 0,
  'log_path': 'C:\\Users\\sjuan\\Desktop\\NLP\\project\\nlp-polimillionaire\\results\\runs\\gemma_4_e2b_20260504_115907.jsonl'},
 {'label': 'Gemma 4 E4B',
  'model_id': 'google/gemma-4-E4B-it',
  'warmup_option': 1,
  'current_level': 2,
  'earned_amount': 100,
  'log_path': 'C:\\Users\\sjuan\\Desktop\\NLP\\project\\nlp-polimillionaire\\results\\runs\\gemma_4_e4b_20260504_115928.jsonl'}]

## Results

Read recent game logs.


In [6]:
from polimillionaire.runner import load_jsonl, summarize_attempts

run_logs = sorted((REPO_ROOT / "results" / "runs").glob("*.jsonl"), key=lambda path: path.stat().st_mtime)

for path in run_logs[-5:]:
    print(path.name)

if run_logs:
    rows = load_jsonl(run_logs[-1])
    print("latest:", run_logs[-1].name)
    print(summarize_attempts(rows))
else:
    print("No logs found.")


gemma_4_e2b_20260504_115318.jsonl
gemma_4_e4b_20260504_115352.jsonl
gemma_4_e2b_20260504_115907.jsonl
gemma_4_e4b_20260504_115928.jsonl
latest: gemma_4_e4b_20260504_115928.jsonl
{'total': 2, 'correct': 1, 'accuracy': 0.5, 'timed_out': 0, 'avg_elapsed_seconds': 1.6639999999999873}


## Done

Use `run_results` and the JSONL logs for comparison.
